In [3]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
import optuna
from optuna.samplers import TPESampler

# 1. 加载并对齐数据
features = pd.read_csv('data/counts_full_mut_features.csv', index_col=0)
labels = pd.read_csv('data/survival_3yr_labels_noNA.csv', index_col=0).iloc[:, 0]
common_idx = features.index.intersection(labels.index)
X = features.loc[common_idx]
y = labels.loc[common_idx]

# 2. 划分训练/测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. 定义 Optuna 目标（最大化 CV AUC）
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_uniform('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-8, 10.0),
        'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-8, 10.0),
        'random_state': 42,
        'use_label_encoder': False,
        'eval_metric': 'logloss'
    }
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('xgb', XGBClassifier(**params))
    ])
    return cross_val_score(
        pipeline, X_train, y_train, 
        cv=5, scoring='roc_auc', n_jobs=-1
    ).mean()

# 4. 运行 Optuna search
study = optuna.create_study(direction='maximize', sampler=TPESampler())
study.optimize(objective, n_trials=50)

# 5. 获取最优参数并重新训练于全训练集
best_params = study.best_trial.params
print("Best parameters:", best_params)
print("Best CV AUC:", study.best_trial.value)

best_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('xgb', XGBClassifier(**best_params))
])
best_pipeline.fit(X_train, y_train)

# 6. 在测试集上评估准确率与 AUC
y_pred = best_pipeline.predict(X_test)
y_proba = best_pipeline.predict_proba(X_test)[:, 1]
print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("Test AUC:", roc_auc_score(y_test, y_proba))


[I 2025-06-09 20:00:36,002] A new study created in memory with name: no-name-abe80ca9-01c7-474d-87fb-aa8d4e1dbfe1
[I 2025-06-09 20:00:41,131] Trial 0 finished with value: 0.6576507370625018 and parameters: {'n_estimators': 191, 'max_depth': 5, 'learning_rate': 0.021421684087086404, 'subsample': 0.8704573974790961, 'colsample_bytree': 0.7714424302578814, 'reg_alpha': 4.955342469808132e-07, 'reg_lambda': 0.0023145922170665673}. Best is trial 0 with value: 0.6576507370625018.
[I 2025-06-09 20:00:44,661] Trial 1 finished with value: 0.6783733695498402 and parameters: {'n_estimators': 351, 'max_depth': 9, 'learning_rate': 0.060237357903495464, 'subsample': 0.6255633458639451, 'colsample_bytree': 0.9429191036141458, 'reg_alpha': 0.2787139171180197, 'reg_lambda': 0.0002506426967406493}. Best is trial 1 with value: 0.6783733695498402.
[I 2025-06-09 20:00:48,417] Trial 2 finished with value: 0.6573451602863368 and parameters: {'n_estimators': 105, 'max_depth': 8, 'learning_rate': 0.040882996811

Best parameters: {'n_estimators': 303, 'max_depth': 6, 'learning_rate': 0.28596456126635905, 'subsample': 0.7363019916066365, 'colsample_bytree': 0.7025484216830613, 'reg_alpha': 0.004971578541816022, 'reg_lambda': 0.02621417426190447}
Best CV AUC: 0.7426180573239398
Test Accuracy: 0.42
Test AUC: 0.43800322061191627


In [7]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

import optuna
from optuna.samplers import TPESampler

# 1. 加载并对齐数据
features = pd.read_csv('data/counts_full_mut_features.csv', index_col=0)
labels   = pd.read_csv('data/survival_3yr_labels_noNA.csv', index_col=0).iloc[:, 0]
common_idx = features.index.intersection(labels.index)
X = features.loc[common_idx]
y = labels.loc[common_idx]

# 2. 划分训练/测试集（80/20，分层抽样）
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. 计算不平衡权重
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# 4. 定义 Optuna 目标：最大化 CV AUC
def objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 50, 200),
        'max_depth':         trial.suggest_int('max_depth', 3, 6),
        'learning_rate':     trial.suggest_loguniform('learning_rate', 0.01, 0.1),
        'subsample':         trial.suggest_uniform('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_uniform('colsample_bytree', 0.6, 1.0),
        'reg_alpha':         trial.suggest_loguniform('reg_alpha', 1e-3, 10.0),
        'reg_lambda':        trial.suggest_loguniform('reg_lambda', 1e-3, 100.0),
        'scale_pos_weight':  scale_pos_weight,
        'random_state':      42,
        'use_label_encoder': False,
        'eval_metric':       'auc'
    }
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('xgb', XGBClassifier(**params))
    ])
    # 5 折 CV，优化 AUC
    return cross_val_score(
        pipeline, X_train, y_train, 
        cv=5, scoring='roc_auc', n_jobs=-1
    ).mean()

# 5. 创建并运行 Optuna study
study = optuna.create_study(direction='maximize', sampler=TPESampler())
study.optimize(objective, n_trials=50)

# 6. 输出最优参数和 CV AUC
best_params = study.best_trial.params
print("最佳参数：", best_params)
print("CV 最佳 AUC：", study.best_trial.value)

# 7. 用最佳参数训练最终模型
final_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('xgb', XGBClassifier(**best_params))
])
final_pipeline.fit(X_train, y_train)

# 8. 在测试集上评估
y_pred  = final_pipeline.predict(X_test)
y_proba = final_pipeline.predict_proba(X_test)[:, 1]

print("测试集 Accuracy：", accuracy_score(y_test, y_pred))
print("测试集 AUC：",       roc_auc_score(y_test, y_proba))


[I 2025-06-09 21:48:19,305] A new study created in memory with name: no-name-7b40e032-0ec5-40a6-b02e-65ae3acd1ac8
[I 2025-06-09 21:48:21,659] Trial 0 finished with value: 0.6932079902668138 and parameters: {'n_estimators': 80, 'max_depth': 3, 'learning_rate': 0.04110350944095157, 'subsample': 0.6861049498926532, 'colsample_bytree': 0.9563896386538749, 'reg_alpha': 1.0777570279181938, 'reg_lambda': 14.884036854325846}. Best is trial 0 with value: 0.6932079902668138.
[I 2025-06-09 21:48:23,972] Trial 1 finished with value: 0.6393642305407011 and parameters: {'n_estimators': 102, 'max_depth': 4, 'learning_rate': 0.06424016159554678, 'subsample': 0.6286007782082764, 'colsample_bytree': 0.7952584218291358, 'reg_alpha': 0.17920125531832057, 'reg_lambda': 0.14010043791270227}. Best is trial 0 with value: 0.6932079902668138.
[I 2025-06-09 21:48:28,130] Trial 2 finished with value: 0.6699841552782729 and parameters: {'n_estimators': 145, 'max_depth': 5, 'learning_rate': 0.03243175337937061, 'su

最佳参数： {'n_estimators': 80, 'max_depth': 3, 'learning_rate': 0.04110350944095157, 'subsample': 0.6861049498926532, 'colsample_bytree': 0.9563896386538749, 'reg_alpha': 1.0777570279181938, 'reg_lambda': 14.884036854325846}
CV 最佳 AUC： 0.6932079902668138
测试集 Accuracy： 0.52
测试集 AUC： 0.537842190016103


In [8]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

import optuna
from optuna.samplers import TPESampler

# 1. 加载并对齐数据
features = pd.read_csv('data/counts_full_mut_features.csv', index_col=0)
labels   = pd.read_csv('data/survival_3yr_labels_noNA.csv', index_col=0).iloc[:, 0]
common_idx = features.index.intersection(labels.index)
X = features.loc[common_idx]
y = labels.loc[common_idx]

# 2. 划分训练/测试集（80/20，分层抽样）
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. 计算不平衡权重
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# 4. 定义 Optuna 目标：多目标优化 AUC & Accuracy
def objective(trial):
    # 超参数空间
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 50, 200),
        'max_depth':         trial.suggest_int('max_depth', 3, 6),
        'learning_rate':     trial.suggest_loguniform('learning_rate', 0.01, 0.1),
        'subsample':         trial.suggest_uniform('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_uniform('colsample_bytree', 0.6, 1.0),
        'reg_alpha':         trial.suggest_loguniform('reg_alpha', 1e-3, 10.0),
        'reg_lambda':        trial.suggest_loguniform('reg_lambda', 1e-3, 100.0),
        'scale_pos_weight':  scale_pos_weight,
        'random_state':      42,
        'use_label_encoder': False,
        'eval_metric':       'auc'
    }
    # 构建流水线
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('xgb', XGBClassifier(**params))
    ])
    # 5 折 CV 分别计算 AUC & Accuracy
    auc = cross_val_score(
        pipeline, X_train, y_train,
        cv=5, scoring='roc_auc', n_jobs=-1
    ).mean()
    acc = cross_val_score(
        pipeline, X_train, y_train,
        cv=5, scoring='accuracy', n_jobs=-1
    ).mean()
    return auc, acc

# 5. 创建并运行 Optuna 多目标 Study
study = optuna.create_study(
    directions=['maximize', 'maximize'],
    sampler=TPESampler()
)
study.optimize(objective, n_trials=50)

# 6. 查看 Pareto 前沿上的最优 Trials
print("Pareto 前沿 Trials:")
for t in study.best_trials:
    print(f"  Trial#{t.number}, AUC={t.values[0]:.4f}, Accuracy={t.values[1]:.4f}")

# 7. 选择其中一个（例如 Accuracy 最大）用于最终训练
best_acc_trial = max(study.best_trials, key=lambda t: t.values[1])
best_params = best_acc_trial.params
print("\n选取最高 Accuracy 的 Trial:", best_acc_trial.number)
print("对应参数：", best_params)

# 8. 用选定参数重训练并评估测试集
final_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('xgb', XGBClassifier(**best_params))
])
final_pipeline.fit(X_train, y_train)

y_pred  = final_pipeline.predict(X_test)
y_proba = final_pipeline.predict_proba(X_test)[:, 1]

print("\n测试集评估：")
print("  Accuracy:", accuracy_score(y_test, y_pred))
print("  AUC     :", roc_auc_score(y_test, y_proba))


[I 2025-06-09 22:00:06,595] A new study created in memory with name: no-name-97477f79-2e90-4b77-9fa2-9f6610caa10b
[I 2025-06-09 22:00:17,580] Trial 0 finished with values: [0.6369549840138076, 0.6121794871794872] and parameters: {'n_estimators': 192, 'max_depth': 5, 'learning_rate': 0.015671825289030904, 'subsample': 0.9119334032745863, 'colsample_bytree': 0.758182162476706, 'reg_alpha': 0.6542857090416012, 'reg_lambda': 0.759410102123212}.
[I 2025-06-09 22:00:22,204] Trial 1 finished with values: [0.626916221033868, 0.6171794871794872] and parameters: {'n_estimators': 51, 'max_depth': 5, 'learning_rate': 0.03349593950809165, 'subsample': 0.7332569468115792, 'colsample_bytree': 0.7246818303078604, 'reg_alpha': 0.001058832517798545, 'reg_lambda': 0.005114918311999139}.
[I 2025-06-09 22:00:28,549] Trial 2 finished with values: [0.6686826245649775, 0.5816666666666667] and parameters: {'n_estimators': 165, 'max_depth': 5, 'learning_rate': 0.010994341061819004, 'subsample': 0.63613611160704

Pareto 前沿 Trials:
  Trial#28, AUC=0.6943, Accuracy=0.6482

选取最高 Accuracy 的 Trial: 28
对应参数： {'n_estimators': 73, 'max_depth': 5, 'learning_rate': 0.019344509205726494, 'subsample': 0.6503902690901734, 'colsample_bytree': 0.9843551884993218, 'reg_alpha': 0.04283456181668269, 'reg_lambda': 6.824239792593356}

测试集评估：
  Accuracy: 0.56
  AUC     : 0.5169082125603865


In [9]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

import optuna
from optuna.samplers import TPESampler

# 1. 加载并对齐数据
features = pd.read_csv('data/counts_full_mut_features.csv', index_col=0)
labels   = pd.read_csv('data/survival_3yr_labels_noNA.csv', index_col=0).iloc[:, 0]
idx      = features.index.intersection(labels.index)
X_all    = features.loc[idx]
y_all    = labels.loc[idx]

# 2. 无监督特征工程
# 2.1 方差过滤
vt   = VarianceThreshold(threshold=1e-5)
X_v  = pd.DataFrame(vt.fit_transform(X_all),
                    index=X_all.index,
                    columns=X_all.columns[vt.get_support()])

# 2.2 相关性过滤
corr  = X_v.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [c for c in upper.columns if (upper[c] > 0.9).any()]
X_f   = X_v.drop(columns=to_drop)

# 2.3 离散 vs 连续
discrete = [c for c in X_f if X_f[c].nunique() <= 10]
continuous = [c for c in X_f if X_f[c].nunique() > 10]

# 2.4 One-Hot 编码离散特征
X_dis = pd.get_dummies(X_f[discrete].astype('category'), drop_first=True)

# 2.5 log1p 变换连续特征
X_cont = np.log1p(X_f[continuous])

# 2.6 合并并准备最终特征矩阵
X_proc = pd.concat([X_cont, X_dis], axis=1)

# 3. 拆分训练/测试集
X_train, X_test, y_train, y_test = train_test_split(
    X_proc, y_all,
    test_size=0.2,
    random_state=42,
    stratify=y_all
)

# 4. 计算不平衡权重
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# 5. 多目标 Optuna 目标函数：同时最大化 AUC 和 Accuracy
def objective(trial):
    # 参数空间
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 50, 200),
        'max_depth':         trial.suggest_int('max_depth', 3, 6),
        'learning_rate':     trial.suggest_loguniform('learning_rate', 0.01, 0.1),
        'subsample':         trial.suggest_uniform('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_uniform('colsample_bytree', 0.6, 1.0),
        'reg_alpha':         trial.suggest_loguniform('reg_alpha', 1e-3, 10.0),
        'reg_lambda':        trial.suggest_loguniform('reg_lambda', 1e-3, 100.0),
        'scale_pos_weight':  scale_pos_weight,
        'random_state':      42,
        'use_label_encoder': False,
        'eval_metric':       'auc'
    }
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('xgb', XGBClassifier(**params))
    ])
    # 5 折 CV 分别测 AUC 和 Accuracy
    auc = cross_val_score(pipe, X_train, y_train,
                          cv=5, scoring='roc_auc', n_jobs=-1).mean()
    acc = cross_val_score(pipe, X_train, y_train,
                          cv=5, scoring='accuracy', n_jobs=-1).mean()
    return auc, acc

# 6. 创建并跑多目标 Study
study = optuna.create_study(directions=['maximize','maximize'],
                            sampler=TPESampler())
study.optimize(objective, n_trials=50)

# 7. 列出 Pareto 前沿上的 Trials
print("Pareto 前沿 Trials:")
for t in study.best_trials:
    print(f"  Trial#{t.number} → AUC={t.values[0]:.4f}, ACC={t.values[1]:.4f}")

# 8. 按需求选出最优 Trial（这里示例选最高 Accuracy）
best_t = max(study.best_trials, key=lambda t: t.values[1])
print("\n选中 Trial#", best_t.number)
print("对应参数：", best_t.params)

# 9. 用选中参数重训练 & 测试评估
final_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('xgb', XGBClassifier(**best_t.params))
])
final_pipe.fit(X_train, y_train)

y_pred  = final_pipe.predict(X_test)
y_proba = final_pipe.predict_proba(X_test)[:, 1]

print("\n测试集评估：")
print("  Accuracy:", accuracy_score(y_test, y_pred))
print("  AUC     :", roc_auc_score(y_test, y_proba))


[I 2025-06-09 22:08:11,765] A new study created in memory with name: no-name-1d120430-702e-4bdb-b343-91b676b7b508
[I 2025-06-09 22:08:13,672] Trial 0 finished with values: [0.6575672693319752, 0.6167948717948718] and parameters: {'n_estimators': 88, 'max_depth': 3, 'learning_rate': 0.08274527062429148, 'subsample': 0.8274545375736931, 'colsample_bytree': 0.7740522573612378, 'reg_alpha': 0.012994112158049125, 'reg_lambda': 0.023383116934032465}.
[I 2025-06-09 22:08:17,548] Trial 1 finished with values: [0.6872365107659226, 0.6426923076923077] and parameters: {'n_estimators': 197, 'max_depth': 3, 'learning_rate': 0.09897005925853022, 'subsample': 0.8955390925512904, 'colsample_bytree': 0.7588266686574238, 'reg_alpha': 0.6395192354073639, 'reg_lambda': 52.544750897296275}.
[I 2025-06-09 22:08:25,310] Trial 2 finished with values: [0.654558186911128, 0.6071794871794872] and parameters: {'n_estimators': 183, 'max_depth': 5, 'learning_rate': 0.06312717484750711, 'subsample': 0.87708304636000

Pareto 前沿 Trials:
  Trial#4 → AUC=0.7071, ACC=0.6583
  Trial#48 → AUC=0.6929, ACC=0.6738

选中 Trial# 48
对应参数： {'n_estimators': 100, 'max_depth': 6, 'learning_rate': 0.029664789945650808, 'subsample': 0.6576590656434091, 'colsample_bytree': 0.9418358518814897, 'reg_alpha': 4.8451066057995105, 'reg_lambda': 0.5354831103007773}

测试集评估：
  Accuracy: 0.48
  AUC     : 0.533011272141707
